[![Jupyter](https://img.shields.io/badge/Jupyter-Notebook-F37626?logo=jupyter&logoColor=white)](#)
[![Python](https://img.shields.io/pypi/pyversions/oracle-vecdb)](https://pypi.org/project/oracle-vecdb/)
[![oracle-vecdb](https://img.shields.io/badge/oracle-vecdb-2EA44F?logo=oracle&logoColor=white)](#)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/vecdb/start_here/quickstart.ipynb)

# Quickstart: Build Vector Search with Python

Build a working vector search application with the Oracle VecDB Python SDK. Configure a client, create an integrated-embedding table, load records, and search by meaning.

## Requirements

- Python 3.10 or later
- Oracle AI Database 23.26.3 or later
- ORDS 26.2.2 or later
- The Oracle VecDB Python SDK installed in this notebook's Python environment

## Before you begin

1. [Choose a database deployment](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/) and complete its setup.
2. Record the vector user name and password, and copy the **SDK REST endpoint**.
3. Load the `all_MiniLM_L12_v2` embedding model using the Vector Database Console. The model name in Section 3 must match the loaded model exactly.

This notebook uses the SDK REST endpoint, not the Console URL.

## 1. Install the SDK

The next cell installs or upgrades the SDK in the Python environment used by this notebook. If you use `uv`, run `uv add oracle-vecdb` in your project instead.

In [ ]:
%%bash
python -m pip install --upgrade oracle-vecdb

## 2. Configure the client

Set `VECDB_REST_URL` and `VECDB_USERNAME` in the environment before starting Jupyter, or edit the two placeholder defaults in the cell. The password is requested securely unless `VECDB_PASSWORD` is already set. For bearer-token authentication, set `VECDB_ACCESS_TOKEN` instead.

In [ ]:
from oracle_vecdb import OracleVecDB, Configuration

config = Configuration(
    rest_url="https://<host>:<port>/ords/<vector_user>/_/db-api/stable/vecdb/",
    username="<VECTOR_USER>",
    password="<VECTOR_USER_PASSWORD>",
)

vecdb = OracleVecDB(config)

# For bearer-token authentication, replace the Configuration block above with:
# config = Configuration(
#     rest_url="https://<host>:<port>/ords/<vector_user>/_/db-api/stable/vecdb/",
#     access_token="<BEARER_TOKEN>",
# )

If you rerun this quickstart and the `demo` table already exists, delete it before continuing. This also deletes its vectors and indexes. Run the next cell only when you intend to reset that table.

In [ ]:
vecdb.drop_vector_table(name="demo")

## 3. Create an integrated embedding vector table

Create a table that generates embeddings from text stored in metadata. The `model` value must match the embedding model loaded during deployment setup.

In [ ]:
vecdb.create_vector_table(
    name="demo",
    embed_params={
        "model": "all_MiniLM_L12_v2",
        "embed_metadata_jsonpath": "content",
    },
)

## 4. Load integrated embedding records

Because the table is configured for integrated embeddings, provide text in the `content` metadata field. The database generates each vector during the upsert.

In [ ]:
vecdb.upsert_vectors(
    table_name="demo",
    vectors=[
        {
            "id": "comedy-1",
            "metadata": {
                "title": "Comedy movie review",
                "content": "A lighthearted comedy filled with witty dialogue and unexpected situations.",
                "genre": "comedy",
            }
        },
        {
            "id": "drama-1",
            "metadata": {
                "title": "Drama movie review",
                "content": "A heartfelt story about relatives overcoming a difficult situation together.",
                "genre": "drama",
            }
        },
        {
            "id": "science-fiction-1",
            "metadata": {
                "title": "Science fiction movie review",
                "content": "A futuristic adventure about explorers traveling beyond the solar system.",
                "genre": "science fiction",
            }
        },
        {
            "id": "documentary-1",
            "metadata": {
                "title": "Documentary movie review",
                "content": "An informative film examining ocean conservation and marine ecosystems.",
                "genre": "documentary",
            }
        },
        {
            "id": "thriller-1",
            "metadata": {
                "title": "Thriller movie review",
                "content": "A suspenseful mystery about an investigator searching for a missing artifact.",
                "genre": "thriller",
            }
        },
    ],
)

## 5. Run a semantic search

The text query is embedded with the table's configured model. It describes the drama record without repeating the words `family` or `drama`.

In [ ]:
results = vecdb.query(
    table_name="demo",
    query_by={"text": "A moving story about people supporting one another through hardship"},
    top_k=3,
)

for item in results.items:
    print(
        f"id: {item.id} | distance: {item.distance} | "
        f"genre: {item.metadata['genre']} | title: {item.metadata['title']}"
    )

The semantically closest result should be `drama-1`, even though the query does not use the words `family` or `drama`. Exact distances and result order can vary with the embedding model and index configuration.

## Next steps

Choose the next task for your application:

- **Search and filter:** explore metadata filters in the [Search Diagnostics notebook](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/vecdb/search/search-diagnostics.ipynb).
- **Ingest and bring your own vectors:** compare integrated embeddings and application-generated vectors in the [Embedding & Ingest Patterns notebook](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/vecdb/embeddings/embedding-ingest-patterns.ipynb).
- **Rerank in the database:** use the [In-database reranking notebook](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/vecdb/reranking/load-model-for-in-database-reranking.ipynb).
- **Rerank with OCI:** use the [OCI Generative AI reranking notebook](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/vecdb/reranking/oci-reranking.ipynb).

For the complete catalog, see the [VecDB sample notebooks](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/vecdb/README.md).